# 02 — EDA & Feature Engineering

This notebook covers:
1. Exploratory data analysis of the curated EGFR dataset
2. Chemical space visualization (PCA/t-SNE on fingerprints)
3. Descriptor distributions and correlations
4. Molecular graph construction examples

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from rdkit import Chem
from rdkit.Chem import Draw

from src.components.feature_engineering import MolecularFeatureEngineer

sns.set_theme(style="whitegrid")
%matplotlib inline

In [ ]:
with open("../configs/config.yaml") as f:
    config = yaml.safe_load(f)

curated_df = pd.read_csv("../data/processed/egfr_curated.csv")
engineer = MolecularFeatureEngineer(config)
print(f"Dataset shape: {curated_df.shape}")
curated_df.head()

## 1. Descriptor Distributions

In [ ]:
desc_df = engineer.compute_descriptors(curated_df["canonical_smiles"].tolist())
desc_with_activity = pd.concat([desc_df, curated_df[["activity_class", "pIC50"]].reset_index(drop=True)], axis=1)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, desc_df.columns):
    sns.histplot(data=desc_with_activity, x=col, hue="activity_class", bins=40, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.savefig("../results/plots/descriptor_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Descriptor Correlation Matrix

In [ ]:
corr = desc_with_activity.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Descriptor Correlation Matrix")
plt.tight_layout()
plt.savefig("../results/plots/descriptor_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Chemical Space Visualization (PCA & t-SNE)

In [ ]:
fp_array = engineer.compute_fingerprints(curated_df["canonical_smiles"].tolist())

# PCA
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(fp_array)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = {"active": "#2ecc71", "inactive": "#e74c3c"}

for cls in ["inactive", "active"]:
    mask = curated_df["activity_class"] == cls
    axes[0].scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                    c=colors[cls], label=cls, alpha=0.5, s=10)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].set_title("PCA of Morgan Fingerprints")
axes[0].legend()

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_coords = tsne.fit_transform(fp_array)

for cls in ["inactive", "active"]:
    mask = curated_df["activity_class"] == cls
    axes[1].scatter(tsne_coords[mask, 0], tsne_coords[mask, 1],
                    c=colors[cls], label=cls, alpha=0.5, s=10)
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].set_title("t-SNE of Morgan Fingerprints")
axes[1].legend()

plt.tight_layout()
plt.savefig("../results/plots/chemical_space.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Build Feature Matrix & Graph Dataset

In [ ]:
# Feature matrix for classical ML
feature_df = engineer.build_feature_matrix(curated_df)
engineer.save_features(feature_df)
print(f"Feature matrix shape: {feature_df.shape}")

# Graph dataset for GNN
graph_dataset = engineer.build_graph_dataset(
    curated_df["canonical_smiles"].tolist(),
    curated_df["pIC50"].tolist()
)
engineer.save_graph_dataset(graph_dataset)
print(f"Graph dataset size: {len(graph_dataset)}")
print(f"Example graph — nodes: {graph_dataset[0].x.shape}, edges: {graph_dataset[0].edge_index.shape}")

## 5. Lipinski Analysis

In [ ]:
lipinski_df = engineer.compute_lipinski(curated_df["canonical_smiles"].tolist())
lipinski_with_activity = pd.concat([lipinski_df, curated_df["activity_class"].reset_index(drop=True)], axis=1)

print(f"Overall Lipinski pass rate: {lipinski_df['Lipinski_Pass'].mean():.1%}")
print(f"Active compounds pass rate: {lipinski_with_activity[lipinski_with_activity['activity_class']=='active']['Lipinski_Pass'].mean():.1%}")
print(f"Inactive compounds pass rate: {lipinski_with_activity[lipinski_with_activity['activity_class']=='inactive']['Lipinski_Pass'].mean():.1%}")